# 📓 Caderno de Exercícios — Aula 03: Limpeza Numérica, Regras Matemáticas e Outliers
**Capacitação de Python para Ciência de Dados — Case Campe Supply**

> 💡 **O Coração Financeiro da Base:** Agora vamos tratar `faturamento`, `margem_de_lucro` e `lucro`. Lembre-se da equação fundamental do negócio:
> $$\text{Lucro} = \text{Faturamento} \times \text{Margem de Lucro}$$

In [1]:
import pandas as pd
import numpy as np
from difflib import get_close_matches

# === Checkpoint Inicial (Trazendo os dados limpos das Aulas 01 e 02) ===
df = pd.read_excel('Campe_Supply_Corrompido.xlsx', sheet_name='Campe Supply')
df = df.drop(columns=['UNNAMED: 0', 'coluna_inutil_0', 'coluna_inutil_1'])
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('ã', 'a')
df = df.drop_duplicates().reset_index(drop=True)

categorias_oficiais = {
    'setor': ['Congelados', 'Latícinio', 'Bebidas', 'Carnes'],
    'produto': ['Queijo', 'Sorvete', 'Frango', 'Refrigerantes', 'Cervejas', 'Bovina', 'Leite', 'Lasanha', 'Pizza', 'Suína', 'Sucos', 'Coalhada'],
    'vendedor': ['Vanessa', 'Amaral', 'Leo', 'Daniele', 'Jonas', 'Evandro'],
    'regiao': ['Sudeste', 'Nordeste', 'Sul'],
    'estado': ['Rio de Janeiro', 'Minas Gerais', 'Bahia', 'Rio Grande do Norte', 'São Paulo', 'Sergipe', 'Rio Grande do Sul', 'Paraná', 'Santa Catarina'],
    'cliente': ['Atacarejo', 'Varejo', 'Atacado']
}
def corrigir_palavra(valor, lista_oficial):
    if pd.isna(valor): return np.nan
    match = get_close_matches(str(valor).strip(), lista_oficial, n=1, cutoff=0.6)
    return match[0] if match else np.nan

for col, lista_valida in categorias_oficiais.items():
    df[col] = df[col].apply(lambda x: corrigir_palavra(x, lista_valida))

mapa_produto_setor = {'Queijo': 'Latícinio', 'Leite': 'Latícinio', 'Coalhada': 'Latícinio', 'Sorvete': 'Congelados', 'Lasanha': 'Congelados', 'Pizza': 'Congelados', 'Frango': 'Carnes', 'Bovina': 'Carnes', 'Suína': 'Carnes', 'Refrigerantes': 'Bebidas', 'Cervejas': 'Bebidas', 'Sucos': 'Bebidas'}
mapa_estado_regiao = {'Rio de Janeiro': 'Sudeste', 'Minas Gerais': 'Sudeste', 'São Paulo': 'Sudeste', 'Bahia': 'Nordeste', 'Rio Grande do Norte': 'Nordeste', 'Sergipe': 'Nordeste', 'Rio Grande do Sul': 'Sul', 'Paraná': 'Sul', 'Santa Catarina': 'Sul'}
df['setor'] = df['setor'].fillna(df['produto'].map(mapa_produto_setor))
df['regiao'] = df['regiao'].fillna(df['estado'].map(mapa_estado_regiao))



df["produto"] = df["produto"].fillna("Não Informado")
df["estado"] = df["estado"].fillna("Não Informado")

df["setor"] = df["setor"].fillna("Não Informado")
df["regiao"] = df["regiao"].fillna("Não Informado")

print('Dados prontos para a Aula 03:', df.shape)



Dados prontos para a Aula 03: (1655, 10)


## 🎯 Missão 1: Investigando e Convertendo as Colunas Numéricas
As colunas `faturamento`, `margem_de_lucro` e `lucro` estão como `object` porque contêm erros de digitação com letras no meio dos números (ex: `"60b.72"`, `"0.a6"`).

1. Filtre ou exiba alguns desses valores corrompidos para ver como eles são.
2. Converta as 3 colunas para tipo numérico (`float`) usando `pd.to_numeric(..., errors='coerce')` (isso transformará os textos inválidos em `NaN` para podermos recalculá-los no próximo passo).
3. Rode um `df[['faturamento', 'margem_de_lucro', 'lucro']].describe()` e observe os valores mínimos e máximos!

In [3]:
# 1. Converta as 3 colunas financeiras para numérico e analise o .describe()
# Seu código aqui:

colunas_financeiras = ['faturamento', 'margem_de_lucro', 'lucro']

for col in colunas_financeiras:
    serie_convertida = pd.to_numeric(df[col], errors='coerce')
    valores_corrompidos = df[df[col].notna() & serie_convertida.isna()][col]
    print(f'Exemplos de textos corrompidos em {col.upper()}(Total: {len(valores_corrompidos)}):')

print(valores_corrompidos.head(5).tolist())
print('-' * 50)

Exemplos de textos corrompidos em FATURAMENTO(Total: 33):
Exemplos de textos corrompidos em MARGEM_DE_LUCRO(Total: 29):
Exemplos de textos corrompidos em LUCRO(Total: 32):
['98.275200000000b1', 't6.32639999999999', 'e46.1072', '72.016100000000h1', '1o2.4068']
--------------------------------------------------


In [4]:
for col in colunas_financeiras:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print('Quantidade de nulos por coluna financeira após conversão:')
print(df[colunas_financeiras].isna().sum())

df[colunas_financeiras].describe()

Quantidade de nulos por coluna financeira após conversão:
faturamento        115
margem_de_lucro    106
lucro              116
dtype: int64


,faturamento,margem_de_lucro,lucro
count,1540.000000,1549.000000,1.539000e+03
mean,896.140506,0.119464,3.542544e+03
std,401.069156,0.043099,9.783214e+04
min,520.360000,0.000000,2.669350e+01
25%,669.647500,0.080000,6.572055e+01
50%,798.740000,0.120000,9.550400e+01
75%,947.420000,0.160000,1.307138e+02
max,8581.600000,0.190000,3.260000e+06


## 🎯 Missão 2: Caçando e Corrigindo os 5 Outliers / Erros Matemáticos
Ao olhar o `.describe()`, você viu coisas absurdas como um **lucro máximo de R$ 3.260.000,00** e um **faturamento de R$ 8.581,60**!

1. Crie um filtro para encontrar todas as linhas onde a conta `faturamento * margem_de_lucro` difere de `lucro` em mais de R$ 0,01.
2. Você vai encontrar exatamente **5 linhas** com erro:
   * Em 2 linhas, o `lucro` está maior que o próprio `faturamento` (milhões). Corrija recalculando `lucro = faturamento * margem_de_lucro`.
   * Em 2 linhas, a `margem_de_lucro` está `0.00`, mas existe lucro positivo. Corrija recalculando `margem_de_lucro = lucro / faturamento`.
   * Em 1 linha, o `faturamento` foi multiplicado por 10 (`8581.60`), mas a margem (`0.06`) e o lucro (`51.4896`) estão certos. Corrija recalculando `faturamento = lucro / margem_de_lucro`.

In [10]:
# 2. Identifique as 5 linhas com erro matemático e corrija cada caso usando a fórmula do negócio
# Seu código aqui:

dif_matematica = (df["faturamento"] * df["margem_de_lucro"]) - df["lucro"].abs()

erros_matematicos = df[dif_matematica.abs() > 0.01]
erros_matematicos[['data_da_venda', 'produto', 'vendedor', 'faturamento', 'margem_de_lucro', 'lucro',]]


,data_da_venda,produto,vendedor,faturamento,margem_de_lucro,lucro
304,2035-03-17,Suína,Amaral,1192.84,0.17,2.027828e+06
572,2035-05-22,Sorvete,Vanessa,8581.60,0.06,5.148960e+01
975,2035-08-25,Lasanha,Amaral,829.00,0.00,1.077700e+02
1304,2035-11-11,Queijo,Amaral,544.25,0.06,3.260000e+06
1426,2035-12-05,Cervejas,Amaral,1804.45,0.00,3.428455e+02


In [14]:
#corrigindo lucros errados 

mask_lucro_errado = (df['lucro'] > df['faturamento'])
df.loc[mask_lucro_errado, 'lucro'] = (df.loc[mask_lucro_errado, 'faturamento'] * df.loc[mask_lucro_errado, 'margem_de_lucro'])

#corrigindo margem 0, se lucro = 0, nao exite lucro, caso contrario lucro existe, recalcular

mask_margem_0 = (df['margem_de_lucro'] == 0) & (df['lucro'] > 0)
df.loc[mask_margem_0, 'margem_de_lucro'] = (df.loc[mask_margem_0, 'lucro'] / df.loc[mask_margem_0, 'faturamento'])

#faturamento acima do normal 

mask_faturamento_errado = (df['faturamento'] * df['margem_de_lucro']-df['lucro']).abs() > 0.1

df.loc[mask_faturamento_errado, 'faturamento'] = (df.loc[mask_faturamento_errado, 'lucro'] / df.loc[mask_faturamento_errado, 'margem_de_lucro'])

erros_restantes = (df['faturamento'] * df['margem_de_lucro'] - df['lucro'].abs() > 0.01).sum()   

print('Quantidade de erros restantes após correção:',(erros_restantes))

Quantidade de erros restantes após correção: 0


## 🎯 Missão 3: Recuperando Nulos Numéricos com Matemática
Agora que não há mais outliers mentirosos, você tem mais de 100 células `NaN` nas 3 colunas numéricas. Porém, na maioria dessas linhas, **2 das 3 colunas estão preenchidas**!

Use a equação $\text{Lucro} = \text{Faturamento} \times \text{Margem}$ para preencher os `NaN` de:
* `lucro` (usando `faturamento * margem_de_lucro`)
* `faturamento` (usando `lucro / margem_de_lucro`)
* `margem_de_lucro` (usando `lucro / faturamento`)

Depois, arredonde as 3 colunas para 2 casas decimais (`.round(2)`).
* ✅ **Checkpoint de Validação:** Os nulos de `faturamento` devem cair de **115 para 20**, os de `margem_de_lucro` de **106 para 16**, e os de `lucro` de **116 para 13**!

In [20]:
# 3. Recupere os valores nulos das 3 colunas financeiras e arredonde para 2 casas decimais
# Seu código aqui:


df['lucro'] = df['lucro'].fillna(df['faturamento'] * df['margem_de_lucro'])
df['faturamento'] = df['faturamento'].fillna(df['lucro'] / df['margem_de_lucro'])
df['margem_de_lucro'] = df['margem_de_lucro'].fillna(df['lucro'] / df['faturamento'])


df['faturamento'] = df['faturamento'].round(2)
df['margem_de_lucro'] = df['margem_de_lucro'].round(2)
df['lucro'] = df['lucro'].round(2)

print('Nulos númericos restantes após recuperação:')
print(df[colunas_financeiras].isna().sum())

df[colunas_financeiras].describe().round(2)


Nulos númericos restantes após recuperação:
faturamento        20
margem_de_lucro    16
lucro              13
dtype: int64


,faturamento,margem_de_lucro,lucro
count,1635.00,1639.00,1642.00
mean,893.15,0.12,106.96
std,348.92,0.04,59.14
min,520.36,0.05,26.69
25%,669.96,0.08,65.83
50%,804.18,0.12,95.66
75%,951.02,0.16,131.08
max,2195.03,0.19,394.82


## 🎯 Missão 4: A Segunda Onda de Duplicatas e Engenharia de Datas
Agora que corrigimos textos (`"Vacessa" -> "Vanessa"`) e números (`"60b.72" -> 604.72`), linhas que eram idênticas — mas estavam disfarçadas por erros de digitação — ficaram iguais!

1. Verifique quantas novas duplicatas surgiram com `df.duplicated().sum()` e remova-as com `df.drop_duplicates()`.
2. Crie uma coluna `mes` extraindo o mês de `data_da_venda` (`df['data_da_venda'].dt.month`).

* ✅ **Checkpoint Final da Limpeza:** Sua base limpa terá exatamente **1.541 linhas únicas**!

In [25]:
# 4. Remova as novas duplicatas reveladas, crie a coluna 'mes' e salve o Excel limpo se desejar
# Seu código aqui:

print('Quantidade de duplicatas antes da remoção:', df.duplicated().sum())

#remocao de duplicatas restantes 

df = df.drop_duplicates().reset_index(drop=True)
print('Formato após remoção de duplicatas:', df.shape)


df['mes'] = df['data_da_venda'].dt.month
df['trimestre'] = df['data_da_venda'].dt.quarter

df.to_excel('Campe_Supply_Limpo.xlsx', index=False)
print('Sucesso! A base Campe_Supply_Limpo.xlsx foi salva')
df.head(10)


Quantidade de duplicatas antes da remoção: 0
Formato após remoção de duplicatas: (1541, 12)
Sucesso! A base Campe_Supply_Limpo.xlsx foi salva


,data_da_venda,setor,produto,vendedor,regiao,estado,cliente,faturamento,margem_de_lucro,lucro,mes,trimestre
0,2035-01-01,Carnes,Suína,Amaral,Sudeste,Minas Gerais,Atacado,633.45,0.13,82.35,1.0,1.0
1,2035-01-01,Latícinio,Queijo,Leo,Sudeste,Minas Gerais,Atacarejo,611.40,0.07,42.80,1.0,1.0
2,2035-01-01,Congelados,Sorvete,Leo,Nordeste,Não Informado,Atacarejo,721.52,0.08,57.72,1.0,1.0
3,2035-01-01,Carnes,Bovina,Daniele,Sudeste,São Paulo,Atacado,679.39,0.09,61.15,1.0,1.0
4,2035-01-01,Carnes,Frango,Vanessa,Sul,Rio Grande do Sul,Varejo,964.52,0.19,183.26,1.0,1.0
5,2035-01-01,Latícinio,Coalhada,Leo,Nordeste,Rio Grande do Norte,Atacarejo,565.70,0.08,45.26,1.0,1.0
6,2035-01-01,Carnes,Frango,Vanessa,Nordeste,Rio Grande do Norte,Atacarejo,721.01,0.05,36.05,1.0,1.0
7,2035-01-01,Carnes,Bovina,Vanessa,Sul,Rio Grande do Sul,Atacado,649.40,0.11,71.43,1.0,1.0
8,2035-01-02,Latícinio,Coalhada,NaN,Sudeste,Minas Gerais,Atacarejo,789.01,0.19,149.91,1.0,1.0
9,NaT,Latícinio,Queijo,Daniele,Nordeste,Bahia,Atacado,574.30,0.05,28.72,NaN,NaN
